# 42 · Residual Transport and Deformation Model

Notebook 41 showed that forced residuals are better described as **condition-dependent deformation** than as a shared aligned subspace.

This notebook tests whether high-condition residuals can be predicted from low-condition residuals plus local features:

\[
r_{\mathrm{high}} \approx f(r_{\mathrm{low}}, X)
\]

rather than from a fixed shared axis.

## Main experiments

1. **Residual transport prediction**
   - linear transport
   - contextual transport
   - nonlinear transport

2. **Transport quality**
   - held-out \(R^2\)
   - transport correlation
   - transport error

3. **Transport-corrected classifier**
   - core only
   - transported residual only
   - core + transported residual

4. **Deformation diagnostics**
   - residual shift size
   - local slope proxy
   - transport gain over shared-subspace baseline

## Goal

Determine whether residual is:
- a shared axis,
- random condition-local noise,
- or a **predictable deformation field**

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)
        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())
    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

## Feature builders

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_windows_and_features(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    stats = {
        "entropy": np.array([window_entropy(w) for w in windows_n]),
        "unevenness": np.array([unevenness(w) for w in windows_n]),
        "ratio_mean": np.array([ratio_mean(w) for w in windows_n]),
        "symmetry": np.array([reversal_symmetry_score(w) for w in windows_n]),
    }

    if feature_family == "minimal":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["ratio_mean"][i]]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "full":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["symmetry"][i], stats["ratio_mean"][i], ratio_std(windows_n[i])]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(feature_family)

    return windows_n, stats, X

def make_contextual_features(X):
    prev_X = np.vstack([X[0], X[:-1]])
    next_X = np.vstack([X[1:], X[-1]])
    delta_prev = X - prev_X
    delta_next = next_X - X
    curv = next_X - 2 * X + prev_X
    return np.hstack([X, delta_prev, delta_next, curv])

def apply_condition_mask(stats, condition_name):
    if condition_name == "low_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] <= thr
    if condition_name == "high_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] > thr
    if condition_name == "low_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] <= thr
    if condition_name == "high_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] > thr
    raise ValueError(condition_name)

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std, mean, std

## Models

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    pos = D[D > 0]
    med = np.median(pos) if len(pos) else 1.0
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :]) ** 2, axis=2)
    return np.exp(-gamma * D2)

def fit_linear_boundary(X0_train, X1_train):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]
    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))
    Xtr, _, mean, std = standardize_train_test(X_train, X_train)
    w = fit_logistic_regression(Xtr, y_train, lr=0.12, n_steps=2400, reg=1e-4)
    return {"kind": "linear", "mean": mean, "std": std, "w": w}

def fit_rbf_boundary(X0_train, X1_train, n_proto=20):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]
    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))
    Xtr, _, mean, std = standardize_train_test(X_train, X_train)
    prototypes = choose_prototypes(Xtr, n_proto=min(n_proto, len(Xtr)))
    gamma = estimate_rbf_gamma(Xtr)
    R_train = rbf_features(Xtr, prototypes, gamma)
    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
    return {"kind": "rbf", "mean": mean, "std": std, "prototypes": prototypes, "gamma": gamma, "w": w}

def score_model(model, X0_test, X1_test):
    m = min(len(X0_test), len(X1_test))
    X0_test = X0_test[:m]
    X1_test = X1_test[:m]
    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))
    Xte = (X_test - model["mean"]) / model["std"]

    if model["kind"] == "linear":
        scores = decision_function_logistic(Xte, model["w"])
    else:
        R_test = rbf_features(Xte, model["prototypes"], model["gamma"])
        scores = decision_function_logistic(R_test, model["w"])

    probs = sigmoid(scores)
    return y_test, scores, probs, X_test

## Metrics and transport helpers

In [ ]:
def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

def evaluate_scores(y_true, scores, probs):
    preds = (probs >= 0.5).astype(int)
    acc = float(np.mean(preds == y_true))
    s0 = scores[y_true == 0]
    s1 = scores[y_true == 1]
    overlap = overlap_coefficient_from_hist(s0, s1, bins=30)
    return {"accuracy": acc, "overlap": overlap}

def fit_linear_regression(X, y, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    I = np.eye(Xb.shape[1]); I[0,0] = 0.0
    beta = np.linalg.solve(Xb.T @ Xb + reg * I, Xb.T @ y)
    return beta

def predict_linear_regression(X, beta):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ beta

def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot <= 1e-12:
        return 0.0
    return float(1.0 - ss_res / ss_tot)

def correlation(x, y):
    x = np.asarray(x); y = np.asarray(y)
    if np.std(x) <= 1e-12 or np.std(y) <= 1e-12:
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])

def fit_transport_linear(r_source, X_source, r_target):
    Z = np.column_stack([r_source, X_source])
    Zs, _, mean, std = standardize_train_test(Z, Z)
    beta = fit_linear_regression(Zs, r_target, reg=1e-3)
    return {"kind": "linear", "mean": mean, "std": std, "beta": beta}

def fit_transport_contextual(r_source, X_source, r_target):
    Xc = make_contextual_features(X_source)
    Z = np.column_stack([r_source, Xc])
    Zs, _, mean, std = standardize_train_test(Z, Z)
    beta = fit_linear_regression(Zs, r_target, reg=1e-3)
    return {"kind": "contextual", "mean": mean, "std": std, "beta": beta}

def fit_transport_nonlinear(r_source, X_source, r_target, n_proto=20):
    Xc = make_contextual_features(X_source)
    base = np.column_stack([r_source, Xc])
    Zs, _, mean, std = standardize_train_test(base, base)
    prototypes = choose_prototypes(Zs, n_proto=min(n_proto, len(Zs)))
    gamma = estimate_rbf_gamma(Zs)
    R = rbf_features(Zs, prototypes, gamma)
    beta = fit_linear_regression(R, r_target, reg=1e-3)
    return {"kind": "nonlinear", "mean": mean, "std": std, "prototypes": prototypes, "gamma": gamma, "beta": beta}

def predict_transport(model, r_source, X_source):
    if model["kind"] == "linear":
        Z = np.column_stack([r_source, X_source])
        Zs = (Z - model["mean"]) / model["std"]
        return predict_linear_regression(Zs, model["beta"])
    elif model["kind"] == "contextual":
        Xc = make_contextual_features(X_source)
        Z = np.column_stack([r_source, Xc])
        Zs = (Z - model["mean"]) / model["std"]
        return predict_linear_regression(Zs, model["beta"])
    else:
        Xc = make_contextual_features(X_source)
        Z = np.column_stack([r_source, Xc])
        Zs = (Z - model["mean"]) / model["std"]
        R = rbf_features(Zs, model["prototypes"], model["gamma"])
        return predict_linear_regression(R, model["beta"])

def residual_only_classifier(r_train, y_train, r_test, y_test):
    Xtr = r_train.reshape(-1, 1)
    Xte = r_test.reshape(-1, 1)
    Xtrs, Xtes, _, _ = standardize_train_test(Xtr, Xte)
    w = fit_logistic_regression(Xtrs, y_train, lr=0.1, n_steps=1600, reg=1e-4)
    scores = decision_function_logistic(Xtes, w)
    probs = predict_proba_logistic(Xtes, w)
    return evaluate_scores(y_test, scores, probs)

def core_plus_signal_classifier(core_train, sig_train, y_train, core_test, sig_test, y_test):
    Xtr = np.column_stack([core_train, sig_train])
    Xte = np.column_stack([core_test, sig_test])
    Xtrs, Xtes, _, _ = standardize_train_test(Xtr, Xte)
    w = fit_logistic_regression(Xtrs, y_train, lr=0.1, n_steps=1800, reg=1e-4)
    scores = decision_function_logistic(Xtes, w)
    probs = predict_proba_logistic(Xtes, w)
    return evaluate_scores(y_test, scores, probs)

## Forced residual extractor

In [ ]:
def extract_forced_residual(X0_low, X1_low, X0_high, X1_high, forcing_mode):
    train0 = np.vstack([X0_low, X0_high])
    train1 = np.vstack([X1_low, X1_high])

    if forcing_mode == "capacity_gap":
        core_model = fit_linear_boundary(train0, train1)
        spec_model = fit_rbf_boundary(train0, train1, n_proto=20)

        low = score_model(core_model, X0_low, X1_low), score_model(spec_model, X0_low, X1_low)
        high = score_model(core_model, X0_high, X1_high), score_model(spec_model, X0_high, X1_high)
        mixed = score_model(core_model, train0, train1), score_model(spec_model, train0, train1)

    elif forcing_mode == "feature_gap":
        ctx_train0, ctx_train1 = make_contextual_features(train0), make_contextual_features(train1)
        ctx_low0, ctx_low1 = make_contextual_features(X0_low), make_contextual_features(X1_low)
        ctx_high0, ctx_high1 = make_contextual_features(X0_high), make_contextual_features(X1_high)

        core_model = fit_rbf_boundary(train0, train1, n_proto=20)
        spec_model = fit_rbf_boundary(ctx_train0, ctx_train1, n_proto=20)

        low = score_model(core_model, X0_low, X1_low), score_model(spec_model, ctx_low0, ctx_low1)
        high = score_model(core_model, X0_high, X1_high), score_model(spec_model, ctx_high0, ctx_high1)
        mixed = score_model(core_model, train0, train1), score_model(spec_model, ctx_train0, ctx_train1)

    elif forcing_mode == "condition_gap":
        core_model = fit_rbf_boundary(train0, train1, n_proto=20)
        spec_model_low = fit_rbf_boundary(X0_low, X1_low, n_proto=20)
        spec_model_high = fit_rbf_boundary(X0_high, X1_high, n_proto=20)
        spec_model_mixed = fit_rbf_boundary(X0_high, X1_high, n_proto=20)

        low = score_model(core_model, X0_low, X1_low), score_model(spec_model_low, X0_low, X1_low)
        high = score_model(core_model, X0_high, X1_high), score_model(spec_model_high, X0_high, X1_high)
        mixed = score_model(core_model, train0, train1), score_model(spec_model_mixed, train0, train1)
    else:
        raise ValueError(forcing_mode)

    def pack(core_out, spec_out):
        y_true, s_core, p_core, X = core_out
        _, s_spec, _, _ = spec_out
        residual_raw = s_spec - s_core
        residual = residual_raw / (np.std(residual_raw) + 1e-6)
        return {
            "y_true": y_true,
            "X": X,
            "s_core": s_core,
            "p_core": p_core,
            "s_spec": s_spec,
            "residual_raw": residual_raw,
            "residual": residual,
        }

    return {"low": pack(*low), "high": pack(*high), "mixed": pack(*mixed)}

## Experiment grid

In [ ]:
window_sizes = [3, 5, 7]
feature_family = "minimal"
sample_size = 100
height_block = (0, 400)

systems = {
    "entropy": ("low_entropy", "high_entropy"),
    "unevenness": ("low_unevenness", "high_unevenness"),
}
tasks = {
    "zeta_vs_GUE": ("zeta", "gue"),
    "zeta_vs_Poisson": ("zeta", "poisson"),
}
forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
transport_modes = ["linear", "contextual", "nonlinear"]

## Base gap slices

In [ ]:
start, stop = height_block
base_gaps = {
    "zeta": zeta_gaps_full[start:stop],
    "poisson": poisson_gaps_full[start:stop],
    "gue": gue_gaps_full[:max(stop - start + 300, 900)],
}

## Conditioned features

In [ ]:
def get_conditioned_features(gaps, condition_name, k=5, feature_family="minimal", n=100):
    _, stats, X = build_windows_and_features(gaps, feature_family=feature_family, k=k)
    mask = apply_condition_mask(stats, condition_name)
    Xc = X[mask]
    return Xc[:min(len(Xc), n)]

conditioned = {}
for k in window_sizes:
    conditioned[k] = {}
    for cond_lo, cond_hi in systems.values():
        for cond in [cond_lo, cond_hi]:
            for ensemble_name, gaps in base_gaps.items():
                conditioned[k][(ensemble_name, cond)] = get_conditioned_features(
                    gaps, cond, k=k, feature_family=feature_family, n=sample_size
                )

## Main transport sweep

In [ ]:
results = []
phase_plane_cache = {}

for system_name, (cond_lo, cond_hi) in systems.items():
    for task_name, (ens0, ens1) in tasks.items():
        for k in window_sizes:
            X0_low = conditioned[k][(ens0, cond_lo)]
            X1_low = conditioned[k][(ens1, cond_lo)]
            X0_high = conditioned[k][(ens0, cond_hi)]
            X1_high = conditioned[k][(ens1, cond_hi)]

            m = min(len(X0_low), len(X1_low), len(X0_high), len(X1_high), sample_size)
            if m < 40:
                continue

            X0_low = X0_low[:m]; X1_low = X1_low[:m]
            X0_high = X0_high[:m]; X1_high = X1_high[:m]

            for forcing_mode in forcing_modes:
                data = extract_forced_residual(X0_low, X1_low, X0_high, X1_high, forcing_mode)

                r_low = data["low"]["residual"]
                r_high = data["high"]["residual"]
                X_low = data["low"]["X"]
                X_high = data["high"]["X"]

                y_low = data["low"]["y_true"]
                y_high = data["high"]["y_true"]
                y_mixed = data["mixed"]["y_true"]

                core_low = data["low"]["s_core"]
                core_high = data["high"]["s_core"]
                core_mixed = data["mixed"]["s_core"]
                mixed_resid = data["mixed"]["residual"]

                base_eval = evaluate_scores(y_mixed, core_mixed, sigmoid(core_mixed))

                shared_baseline = 0.5 * (r_low + r_high)
                shared_mixed = np.interp(
                    np.linspace(0, len(shared_baseline)-1, len(mixed_resid)),
                    np.arange(len(shared_baseline)),
                    shared_baseline
                )
                shared_model_eval = core_plus_signal_classifier(
                    np.concatenate([core_low, core_high]),
                    np.concatenate([shared_baseline, shared_baseline]),
                    np.concatenate([y_low, y_high]),
                    core_mixed,
                    shared_mixed,
                    y_mixed,
                )
                shared_gain = shared_model_eval["overlap"] - base_eval["overlap"]

                for transport_mode in transport_modes:
                    if transport_mode == "linear":
                        tmodel = fit_transport_linear(r_low, X_low, r_high)
                    elif transport_mode == "contextual":
                        tmodel = fit_transport_contextual(r_low, X_low, r_high)
                    else:
                        tmodel = fit_transport_nonlinear(r_low, X_low, r_high, n_proto=20)

                    pred_high = predict_transport(tmodel, r_low, X_low)
                    transport_r2 = r2_score(r_high, pred_high)
                    transport_corr = correlation(r_high, pred_high)
                    transport_mae = float(np.mean(np.abs(r_high - pred_high)))

                    transported_mixed = np.interp(
                        np.linspace(0, len(pred_high)-1, len(mixed_resid)),
                        np.arange(len(pred_high)),
                        pred_high
                    )

                    transport_only = residual_only_classifier(pred_high, y_high, transported_mixed, y_mixed)
                    core_plus_transport = core_plus_signal_classifier(
                        core_high,
                        pred_high,
                        y_high,
                        core_mixed,
                        transported_mixed,
                        y_mixed,
                    )

                    deformation_shift = float(np.mean(np.abs(r_high - r_low)))
                    local_slope_proxy = float(np.mean(np.abs(np.diff(pred_high)))) if len(pred_high) > 1 else 0.0

                    results.append({
                        "system": system_name,
                        "task": task_name,
                        "k": k,
                        "forcing_mode": forcing_mode,
                        "transport_mode": transport_mode,
                        "transport_r2": transport_r2,
                        "transport_corr": transport_corr,
                        "transport_mae": transport_mae,
                        "deformation_shift": deformation_shift,
                        "local_slope_proxy": local_slope_proxy,
                        "core_overlap": base_eval["overlap"],
                        "shared_gain": float(shared_gain),
                        "transport_only_overlap": transport_only["overlap"],
                        "core_plus_transport_overlap": core_plus_transport["overlap"],
                        "transport_gain": float(core_plus_transport["overlap"] - base_eval["overlap"]),
                    })

                    if (
                        system_name == "entropy"
                        and task_name == "zeta_vs_GUE"
                        and forcing_mode == "condition_gap"
                        and transport_mode == "nonlinear"
                        and k == 5
                    ):
                        phase_plane_cache["example"] = {
                            "core_mixed": core_mixed,
                            "transported_mixed": transported_mixed,
                            "y_mixed": y_mixed,
                        }

len(results)

## Plot 1 · Transport prediction quality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

for ax, metric, title in zip(
    axes,
    ["transport_r2", "transport_corr"],
    ["Transport $R^2$", "Transport correlation"]
):
    for transport_mode in transport_modes:
        rows = [r for r in results if r["transport_mode"] == transport_mode and r["forcing_mode"] == "condition_gap" and r["task"] == "zeta_vs_GUE" and r["system"] == "entropy"]
        rows = sorted(rows, key=lambda x: x["k"])
        ax.plot([r["k"] for r in rows], [r[metric] for r in rows], marker="o", label=transport_mode)
    ax.set_title(title)
    ax.set_xlabel("window size k")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Plot 2 · Transport heatmaps

In [ ]:
def build_matrix(metric, system_name, forcing_mode):
    rows = [r for r in results if r["system"] == system_name and r["task"] == "zeta_vs_GUE" and r["forcing_mode"] == forcing_mode]
    return np.array([
        [next(r for r in rows if r["k"] == k and r["transport_mode"] == tm)[metric] for k in window_sizes]
        for tm in transport_modes
    ])

fig, axes = plt.subplots(3, 2, figsize=(10, 11), sharex=True, sharey=True)
for i, forcing_mode in enumerate(forcing_modes):
    for j, system_name in enumerate(["entropy", "unevenness"]):
        ax = axes[i, j]
        M = build_matrix("transport_r2", system_name, forcing_mode)
        im = ax.imshow(M, aspect="auto", origin="upper")
        ax.set_title(f"{forcing_mode} · {system_name}")
        ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
        ax.set_yticks(np.arange(len(transport_modes)), transport_modes)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="transport $R^2$")
plt.tight_layout()
plt.show()

## Plot 3 · Transport gain over shared baseline

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    x = np.arange(len(window_sizes))
    width = 0.22
    for offset, transport_mode in zip([-width, 0, width], transport_modes):
        rows = [r for r in results if r["system"] == system_name and r["task"] == "zeta_vs_GUE" and r["forcing_mode"] == "condition_gap" and r["transport_mode"] == transport_mode]
        rows = sorted(rows, key=lambda x: x["k"])
        vals = [r["transport_gain"] - r["shared_gain"] for r in rows]
        ax.bar(x + offset, vals, width, label=transport_mode)
    ax.set_xticks(x, window_sizes)
    ax.set_title(f"{system_name}: condition_gap")
    ax.set_xlabel("window size k")
    ax.set_ylabel("transport gain - shared gain")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Plot 4 · Core vs core+transport

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)

for ax, forcing_mode in zip(axes, forcing_modes):
    rows = [r for r in results if r["forcing_mode"] == forcing_mode and r["transport_mode"] == "nonlinear" and r["system"] == "entropy" and r["task"] == "zeta_vs_GUE"]
    rows = sorted(rows, key=lambda x: x["k"])
    ax.plot(window_sizes, [r["core_overlap"] for r in rows], marker="o", label="core")
    ax.plot(window_sizes, [r["transport_only_overlap"] for r in rows], marker="o", label="transport-only")
    ax.plot(window_sizes, [r["core_plus_transport_overlap"] for r in rows], marker="o", label="core+transport")
    ax.set_title(forcing_mode)
    ax.set_xlabel("window size k")
    ax.legend(fontsize=8)

axes[0].set_ylabel("overlap")
plt.tight_layout()
plt.show()

## Plot 5 · Deformation diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=False)

for ax, metric, title in zip(
    axes,
    ["deformation_shift", "local_slope_proxy"],
    ["Residual shift size", "Local slope proxy"]
):
    for transport_mode in transport_modes:
        rows = [r for r in results if r["transport_mode"] == transport_mode and r["forcing_mode"] == "condition_gap" and r["task"] == "zeta_vs_GUE" and r["system"] == "entropy"]
        rows = sorted(rows, key=lambda x: x["k"])
        ax.plot([r["k"] for r in rows], [r[metric] for r in rows], marker="o", label=transport_mode)
    ax.set_title(title)
    ax.set_xlabel("window size k")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Plot 6 · Transport phase plane

In [ ]:
ex = phase_plane_cache["example"]
fig, ax = plt.subplots(figsize=(6.4, 5.2))

mask0 = ex["y_mixed"] == 0
mask1 = ex["y_mixed"] == 1

ax.scatter(ex["core_mixed"][mask0], ex["transported_mixed"][mask0], s=10, alpha=0.6, label="class 0")
ax.scatter(ex["core_mixed"][mask1], ex["transported_mixed"][mask1], s=10, alpha=0.6, label="class 1")
ax.set_xlabel("core score")
ax.set_ylabel("transported residual proxy")
ax.set_title("Transport phase plane · condition_gap · nonlinear · entropy · zeta_vs_GUE · k=5")
ax.legend()
plt.tight_layout()
plt.show()

## Summary table

In [ ]:
summary_rows = []
for forcing_mode in forcing_modes:
    for system_name in ["entropy", "unevenness"]:
        for task_name in ["zeta_vs_GUE", "zeta_vs_Poisson"]:
            for k in window_sizes:
                candidates = [r for r in results if r["forcing_mode"] == forcing_mode and r["system"] == system_name and r["task"] == task_name and r["k"] == k]
                best = max(candidates, key=lambda r: r["transport_gain"])
                if best["transport_r2"] > 0.2 and best["transport_gain"] > best["shared_gain"]:
                    verdict = "predictable deformation field"
                elif best["transport_r2"] > 0.05:
                    verdict = "weak transport structure"
                else:
                    verdict = "mostly condition-local deformation"
                summary_rows.append({
                    "forcing_mode": forcing_mode,
                    "system": system_name,
                    "task": task_name,
                    "k": k,
                    "best_transport_mode": best["transport_mode"],
                    "transport_r2": best["transport_r2"],
                    "transport_corr": best["transport_corr"],
                    "shared_gain": best["shared_gain"],
                    "transport_gain": best["transport_gain"],
                    "core_plus_transport_overlap": best["core_plus_transport_overlap"],
                    "verdict": verdict,
                })
summary_rows

## Compact summary

In [ ]:
summary = {
    "window_sizes": window_sizes,
    "forcing_modes": forcing_modes,
    "transport_modes": transport_modes,
    "systems": list(systems.keys()),
    "tasks": list(tasks.keys()),
    "summary_rows": summary_rows,
}
summary

## Reading guide

- **Transport $R^2$** asks whether high-condition residual can be predicted from low-condition residual plus features.
- **Transport correlation** asks whether the mapped residual follows the target shape.
- **Transport gain - shared gain** asks whether deformation modeling beats the shared-subspace baseline.
- **Core vs core+transport** asks whether transported residual improves classification beyond the invariant core.
- **Deformation diagnostics** summarize the size and roughness of the learned transport field.

## Verdicts

- **predictable deformation field** → transport works and outperforms the shared baseline
- **weak transport structure** → some predictability, but limited classifier value
- **mostly condition-local deformation** → transport fails, residual remains local